# Neo4j AuraDB — POC Jumpstart Walkthrough

This notebook walks through the import pipeline step by step so you can verify connectivity, inspect your data, and run a test import before committing to a full load.

**Steps:**
1. Install dependencies
2. Load credentials from `.env`
3. Verify AuraDB connectivity
4. Preview a sample batch from your source
5. Create indexes and constraints
6. Run a small test import
7. Run the full import

> **Private-link users:** If `verify_connectivity()` fails, confirm your private DNS alias resolves from this environment before proceeding. On Vertex AI Notebooks, check that your VPC peering to the AuraDB region is active.

---

### 📚 New to Neo4j? Start here first.

These free GraphAcademy courses cover the concepts used throughout this notebook — each is hands-on, browser-based, and takes 1–2 hours:

| Concept | Course | Time |
|---|---|---|
| Graph databases & Neo4j basics | [Neo4j Fundamentals](https://graphacademy.neo4j.com/courses/neo4j-fundamentals/) | 1 hr |
| AuraDB setup & cloud basics | [AuraDB Fundamentals](https://graphacademy.neo4j.com/courses/aura-fundamentals/) | 1 hr |
| Cypher query language | [Cypher Fundamentals](https://graphacademy.neo4j.com/courses/cypher-fundamentals/) | 1 hr |
| Graph data modeling | [Graph Data Modeling Fundamentals](https://graphacademy.neo4j.com/courses/modeling-fundamentals/) | 2 hr |
| Cypher Indexes and Constraints | [Cypher Indexes and Constraints](https://graphacademy.neo4j.com/courses/cypher-indexes-constraints/?category=processing/) | 2-3 hr |
| Importing data into Neo4j | [Importing Data Fundamentals](https://graphacademy.neo4j.com/courses/importing-fundamentals/) | 2 hr |
| Using Neo4j from Python | [Using Neo4j with Python](https://graphacademy.neo4j.com/courses/drivers-python/) | 1 hr |

## 1. Install Dependencies

In [ ]:
# Run once. Restart the kernel after installing.
%pip install neo4j python-dotenv google-cloud-bigquery google-cloud-storage db-dtypes --quiet

# Optional: install boto3 for HMAC / S3-interoperability auth (no ADC required).
# Only needed if you set GCP_HMAC_ACCESS_KEY / GCP_HMAC_SECRET_KEY in .env.
# %pip install boto3 --quiet

## 2. Load Credentials

Copy `env.sample` to `.env` in this directory and fill in your values before running this cell.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Confirm variables loaded — values are masked for safety
required = ["NEO4J_URI", "NEO4J_USER", "NEO4J_PASSWORD"]
for var in required:
    val = os.getenv(var)
    if val:
        masked = val[:8] + "***" if len(val) > 8 else "***"
        print(f"  {var} = {masked}")
    else:
        print(f"  ⚠️  {var} is NOT set")

## 3. Verify AuraDB Connectivity

This will immediately surface any private-link, DNS, or credential issues — before you attempt an import.

In [ ]:
from importer import Neo4jImporter

try:
    importer = Neo4jImporter()  # verify_connectivity() is called inside __init__
    print("✅ Connected successfully.")
except ConnectionError as e:
    print(f"❌ Connection failed:\n{e}")

## Routing Table Diagnostic

Confirms that the driver can see a writable server in the cluster routing table. Useful for catching cross-region PSC issues before a write attempt.

In [ ]:
with importer.driver.session() as session:
    print("Checking Routing Table...")
    result = session.run("CALL dbms.cluster.routing.getRoutingTable({}, 'neo4j')")
    
    for record in result:
        print(f"TTL: {record['ttl']}")
        for server in record['servers']:
            role = server['role']
            addresses = server['addresses']
            print(f"  Role: {role:7} | Addresses: {addresses}")

# Logic to flag the any neo4j routing challenges (this can happen due to connectivity between this notebook and your AuraDB instance)
if not any(s['role'] == 'WRITE' for s in record['servers']):
    print("\n⚠️  DIAGNOSTIC ALERT: No WRITE servers found in routing table.")
    print("This could indicate a cross-region PSC routing issue, or other firewall/security constraints. Please check 'Global Access' on the PSC endpoint if attempting cross-region access.")

## 4. Preview a Sample Batch

Inspect your data before writing anything to the graph. Adjust the query and source below to match your dataset.

In [ ]:
from sources.bigquery_source import BigQuerySource
# from sources.gcs_source import GCSSource  # uncomment if using GCS

# ✏️  Replace with your query / source
PREVIEW_QUERY = """
    SELECT id, name, email
    FROM `your-project.your-dataset.your-table`
    LIMIT 10
"""

source = BigQuerySource(PREVIEW_QUERY)

# Pull just the first batch and display it
first_batch = next(source.get_batches(batch_size=10))

print(f"Rows returned: {len(first_batch)}")
print(f"Columns: {list(first_batch[0].keys())}\n")
for row in first_batch:
    print(row)

### Optional: Apply Your Transform and Re-preview

If you have a `transform_fn`, verify it here before the import runs.

In [ ]:
from main import transform_part_row  # or define a lambda inline

transformed = [transform_part_row(row) for row in first_batch]
transformed = [r for r in transformed if r is not None]  # drop skipped rows

print(f"Rows after transform (skipped {len(first_batch) - len(transformed)}):")
for row in transformed:
    print(row)

## 5. Indexes and Constraints

Before importing, you need to tell Neo4j how to find nodes efficiently during a `MERGE`. Without this, every `MERGE` performs a full label scan — at POC scale it's slow, at real scale it can crater the import entirely.

You have two tools for this: **indexes** and **constraints**. They're related but not interchangeable.

---

### `CREATE INDEX` vs. `CREATE CONSTRAINT`

| | `CREATE INDEX` | `CREATE CONSTRAINT` (uniqueness) |
|---|---|---|
| **Speeds up MERGE** | ✅ Yes | ✅ Yes (constraint implies an index) |
| **Prevents duplicates** | ❌ No | ✅ Yes — write will error if violated |
| **When to use** | Properties queried often, but duplicates are acceptable | Properties that must be unique across all nodes of that label |
| **On violation** | Silent — no error, duplicates can accumulate | Hard error — transaction rolls back |

**Rule of thumb:**
- If a property is the *identity* of a node (a natural key — user ID, product SKU, supplier code), use a **constraint**. You get the index performance *and* a safety net that prevents silent duplicates if something goes wrong upstream.
- If a property is just *frequently searched* but not required to be unique, use a plain **index**.

> **⚠️ Important:** A constraint only enforces uniqueness for nodes of that specific label. Two nodes with *different labels* can share the same property value without violating the constraint — which is usually the right behaviour.

> 📚 **Want to go deeper on Indexing?** [Cypher Fundamentals](https://graphacademy.neo4j.com/courses/cypher-indexes-constraints/?category=processing) covers `Constraints`, `Indexes`, `full-text indexes`, and how and when to yse them to improve query and load performance — free, 1 hour, no installation required.

---

### `CREATE` vs. `MERGE` — and why idempotency matters

This is one of the most common sources of confusion when getting started with Cypher.

- **`CREATE`** always creates a new node or relationship, regardless of whether one already exists. Run it twice, you get two nodes. This is rarely what you want during a batch import.
- **`MERGE`** checks first: if a node matching the pattern exists, it uses it; if not, it creates one. Run it a hundred times, you get one node. This is **idempotent** — safe to re-run without producing duplicates.

For `MERGE` to work correctly, two things must be true:
1. The property you're merging on must reliably identify the node — don't merge on a nullable (will fail) or non-unique field.
2. An index (or uniqueness constraint) must exist on that property — otherwise Neo4j has to scan every node of that label to check for a match, which is very slow at scale.

All Cypher in this toolkit uses `MERGE` for exactly this reason. The `ON CREATE SET` / `ON MATCH SET` pattern lets you set different properties depending on whether the node was just created or already existed:

```cypher
MERGE (p:ClientNode {id: row.id})        -- match or create on the identity key
ON CREATE SET p.createdAt = now          -- only runs on first insert
ON MATCH SET  p.updatedAt = now          -- only runs on subsequent runs
SET p.name = row.name                    -- always applied
```

> 📚 **Want to go deeper on Cypher?** [Cypher Fundamentals](https://graphacademy.neo4j.com/courses/cypher-fundamentals/) covers `MERGE`, `MATCH`, `CREATE`, and the `UNWIND` pattern used throughout this toolkit — free, 1 hour, no installation required.

---

### Creating Indexes and Constraints

Run these in the **AuraDB Browser** (or via a setup script) before running your import. The `IF NOT EXISTS` clause makes them safe to re-run.

In [ ]:
-- Uniqueness constraints: use these for identity properties (natural keys).
-- A constraint automatically creates a backing index, so you get both
-- duplicate prevention and MERGE performance in one statement.
CREATE CONSTRAINT client_node_id_unique IF NOT EXISTS
  FOR (p:ClientNode) REQUIRE p.id IS UNIQUE;

CREATE CONSTRAINT supplier_code_unique IF NOT EXISTS
  FOR (d:Supplier) REQUIRE d.supplierCode IS UNIQUE;

-- Plain index: use this for properties that are queried/filtered often
-- but don't need to be globally unique (e.g. status, region, category).
-- CREATE INDEX client_node_email IF NOT EXISTS FOR (p:ClientNode) ON (p.email);

Verify that your indexes and constraints are active before starting the import:

> 📚 **Indexes and constraints in depth:** [Cypher Indexes and Constraints](https://graphacademy.neo4j.com/courses/cypher-indexes-constraints/) — covers range indexes, full-text indexes, composite keys, and constraint types. 2–3 hours.

In [ ]:
SHOW INDEXES;
SHOW CONSTRAINTS;

## 6. Test Import (Small Batch)

Run a limited import first — verify node/relationship counts look right before going full scale.

> 📚 **About the import pattern used here:** [Importing Data Fundamentals](https://graphacademy.neo4j.com/courses/importing-fundamentals/) and [Importing CSV Data into Neo4j](https://graphacademy.neo4j.com/courses/importing-cypher/) both cover the `UNWIND $rows AS row` batching pattern in detail.

In [ ]:
# ✏️  Replace with your Cypher and source
TEST_QUERY = """
    SELECT id, name, email
    FROM `your-project.your-dataset.your-table`
    LIMIT 50
"""

NODE_CYPHER = """
UNWIND $rows AS row
WITH row, datetime() AS now
MERGE (p:ClientNode {id: row.id})
ON CREATE SET p.createdAt = now, p.updatedAt = now
ON MATCH SET  p.updatedAt = now
SET p.name = row.name, p.email = row.email
"""

test_source = BigQuerySource(TEST_QUERY)
totals = importer.run_import(test_source, NODE_CYPHER, batch_size=50)

print("\n--- Test Import Summary ---")
print(f"  Nodes created:        {totals['nodes_created']}")
print(f"  Properties set:       {totals['properties_set']}")
print(f"  Relationships created: {totals['relationships_created']}")

### Verify in Neo4j

Run this cell to confirm the test nodes landed correctly.

In [ ]:
with importer.driver.session() as session:
    result = session.run("""
        MATCH (p:ClientNode)
        RETURN count(p) AS total, 
               min(p.createdAt) AS earliest,
               max(p.createdAt) AS latest
    """)
    record = result.single()
    print(f"  Total ClientNodes in graph: {record['total']}")
    print(f"  Earliest createdAt:         {record['earliest']}")
    print(f"  Latest createdAt:           {record['latest']}")

## 7. Full Import

Once the test looks good, run the full import. The importer logs progress per batch and prints a final summary.

In [ ]:
# ✏️  Replace with your full query and batch size
FULL_QUERY = """
    SELECT id, name, email
    FROM `your-project.your-dataset.your-table`
"""

full_source = BigQuerySource(FULL_QUERY)
totals = importer.run_import(full_source, NODE_CYPHER, batch_size=1000)

print("\n--- Full Import Summary ---")
print(f"  Nodes created:         {totals['nodes_created']}")
print(f"  Properties set:        {totals['properties_set']}")
print(f"  Relationships created: {totals['relationships_created']}")

## 8. Cleanup

Always close the driver when you're done.

In [ ]:
importer.close()
print("Driver closed.")

---

## Appendix: GCS CSV Import

Swap in `GCSSource` if your data lives in a GCS bucket rather than a BigQuery table.
Everything else is identical — same `get_batches()` interface, same Cypher, same importer.

`GCSSource` supports two auth paths, selected automatically from `.env`:

| Path | When to use | Env vars |
|---|---|---|
| **ADC** (default) | Service account JSON, gcloud creds, Workload Identity | `GOOGLE_APPLICATION_CREDENTIALS` |
| **HMAC** | No ADC available — on-prem, certain CI runners | `GCP_HMAC_ACCESS_KEY` + `GCP_HMAC_SECRET_KEY` |

### ADC path (Application Default Credentials)


In [ ]:
from sources.gcs_source import GCSSource

SUPPLIER_CYPHER = """
UNWIND $rows AS row
MERGE (d:Supplier {supplierCode: row.code})
SET d.name = row.name, d.location = row.city
"""

# ✏️  Replace bucket_name and blob_name with your values
gcs_source = GCSSource(bucket_name="my-data-landing", blob_name="suppliers.csv")

# Preview first
first_batch = next(gcs_source.get_batches(batch_size=5))
print(f"Columns: {list(first_batch[0].keys())}")
for row in first_batch:
    print(row)

In [ ]:
# Run the GCS import
with Neo4jImporter() as gcs_importer:
    totals = gcs_importer.run_import(
        source=gcs_source,
        cypher_query=SUPPLIER_CYPHER,
        batch_size=500,
        transform_fn=lambda r: {**r, "name": r["name"].strip()},
    )
    print(f"\nSuppliers merged: {totals['nodes_created']} created, "
          f"{totals['properties_set']} properties set")

### HMAC Path (S3-interoperability)

Use this when your environment has no ADC credentials — no service account JSON
or gcloud metadata server required.  `GCSSource` detects the HMAC keys
automatically from `.env`; no code change is needed.

**One-time setup:**
1. Generate an HMAC key pair: *GCS Console → Settings → Interoperability tab →
   Service account HMAC keys*. The access key ID will start with `GOOG`.
2. Add the two values to your `.env` (see `env.sample` for the variable names).
3. Install `boto3` (once per environment — cell below).


In [ ]:
# Install the optional HMAC extra (run once, then restart kernel)
%pip install boto3 --quiet


In [ ]:
# Verify HMAC credentials loaded from .env
import os
from dotenv import load_dotenv
load_dotenv(override=True)

hmac_key    = os.getenv("GCP_HMAC_ACCESS_KEY")
hmac_secret = os.getenv("GCP_HMAC_SECRET_KEY")

if hmac_key and hmac_secret:
    print(f"  GCP_HMAC_ACCESS_KEY = {hmac_key[:8]}***")
    print(f"  GCP_HMAC_SECRET_KEY = ***")
    print("  HMAC keys loaded — GCSSource will use S3-interoperability auth.")
else:
    print("  ⚠️  HMAC keys not set.")
    print("  Set GCP_HMAC_ACCESS_KEY and GCP_HMAC_SECRET_KEY in .env, then re-run.")


In [ ]:
from sources.gcs_source import GCSSource

# GCSSource auto-selects HMAC when both keys are present in .env.
# The get_batches() interface is identical — no code change needed.

# ✏️  Replace with your bucket and blob values
hmac_source = GCSSource(bucket_name="my-data-landing", blob_name="suppliers.csv")

# Preview the first batch
first_batch = next(hmac_source.get_batches(batch_size=5))
print(f"Columns: {list(first_batch[0].keys())}")
for row in first_batch:
    print(row)


In [ ]:
# Run the HMAC-authenticated import — identical to the ADC path
from importer import Neo4jImporter

SUPPLIER_CYPHER = """
UNWIND $rows AS row
MERGE (d:Supplier {supplierCode: row.code})
SET d.name = row.name, d.location = row.city
"""

with Neo4jImporter() as hmac_importer:
    totals = hmac_importer.run_import(
        source=hmac_source,
        cypher_query=SUPPLIER_CYPHER,
        batch_size=500,
    )
    print(f"\nSuppliers merged: {totals['nodes_created']} created, "
          f"{totals['properties_set']} properties set")


---

## Using AI to Add a New Connector

The repo is designed so that an AI coding agent can implement a new data source
connector with minimal hand-holding, as long as it's pointed at the right context first.

### Prerequisites
- Repo cloned, venv active (see Quickstart in README)
- `.env` configured with AuraDB credentials
- An AI coding agent: Claude Code (recommended), GitHub Copilot Workspace,
  or ChatGPT / Codex with file upload

---

### Workflow

#### Step 1 — Orient the agent
Point it at the session file before writing a single line of code:

> "Read `.session/add-connector.md` completely, then implement a connector
>  for [source system]. Ask me for any credential or schema details you need."

#### Step 2 — Name your connector and describe your source
Tell the agent the source system (e.g. Snowflake, Databricks, Oracle, S3, Postgres)
and provide any connection credential shape or SDK name you know.

#### Step 3 — Review the generated code
Before running anything, verify:
- [ ] New file is at `sources/<source_name>_source.py`
- [ ] `get_batches()` **yields lists of dicts** (not a single dict, not a generator of scalars)
- [ ] Class is imported and an `elif` branch added in `build_source()` in `main.py`
- [ ] A new entry exists in `config.yaml` with valid Cypher using `UNWIND $rows AS row`
- [ ] An index exists for any property used in a `MERGE` clause (see Indexing section above)

#### Step 4 — Test with a small batch
Set `batch_size: 10` in `config.yaml` for the new job and run:
```python
from main import run_poc
run_poc("config.yaml")
```
Check the summary log output. Fix any errors before increasing batch size.

#### Step 5 — Open a PR
Conventional commit prefix: `feat(sources): add <source_name> connector`

---

### Tool-Specific Tips

#### Claude Code
Paste this as your opening message in a Claude Code session:

```
Read .session/add-connector.md completely, then implement a connector for [source system].
Ask me for any credential structure or SDK details before writing code.
```

Claude Code can read repo files directly — give it access to the full working directory.

#### GitHub Copilot Workspace
- Open a new Workspace task; paste the **Goal** section of `add-connector.md` as the task description.
- Attach `sources/base.py` and `sources/bigquery_source.py` as context files.
- Use the "Generate plan" step and review before accepting code.

#### ChatGPT / Codex with file upload
- Upload `sources/base.py` and `sources/bigquery_source.py`.
- Paste the **Instructions** section from `.session/add-connector.md` verbatim as your first message.
- Ask for one file at a time to stay within context limits.

---

### What the AI Should NOT Change
- `sources/base.py` — the abstract interface is frozen
- `importer.py` — do not touch
- Existing source implementations

### Common Failures
| Symptom | Fix |
|---|---|
| Agent modified `sources/base.py` | Re-prompt: "Do not modify `sources/base.py`." Revert the file and retry. |
| `get_batches()` yields single dicts instead of lists | Re-prompt: "Each `yield` must produce a **list** of dicts, not a single dict." |
| New source never runs | Check `main.py` — the source must be imported and an `elif` branch added to `build_source()`. There is no auto-discovery. |
| MERGE is slow or times out | Create an index on the MERGE property before running the import. See the Indexing section above. |
| Agent adds unknown pip packages | Review dependencies in generated code; install manually and note in your PR description. |
